In [1]:
!pip install langchain --quiet
!pip install langchain-core --quiet
!pip install langchain-community --quiet
!pip install pypdf --quiet
!pip install pytesseract --quiet
!pip install pillow --quiet
!pip install pdf2image --quiet
!brew install tesseract --quiet 2>/dev/null || echo "Tesseract may need manual install"
!brew install poppler --quiet 2>/dev/null || echo "Poppler may need manual install"

⠋ JSON API formula.jws.json                          Downloading  33.2MB/-------
⠋ JSON API cask.jws.json                             Downloading  16.9MB/-------⠋ JSON API formula.jws.json                          Downloading  33.2MB/-------
⠋ JSON API cask.jws.json                             Downloading  16.9MB/-------⠙ JSON API formula.jws.json                          Downloading  33.2MB/-------
⠚ JSON API cask.jws.json                             Downloaded   16.9MB/-------✔︎ JSON API formula.jws.json                          Downloaded   33.2MB/ 33.2MB
✔︎ JSON API cask.jws.json                             Downloaded   16.9MB/ 16.9MB


In [2]:
import pytesseract
import os
from PIL import Image
from langchain_community.document_loaders import PyPDFLoader, YoutubeLoader
from langchain_core.documents import Document


def load_pdf_with_ocr(pdf_path):
    """
    Load PDF and extract text from both text-based and image-based pages using OCR.
    Falls back to OCR if standard text extraction returns empty content.
    """
    documents = []

    # First try standard PDF loader
    try:
        loader = PyPDFLoader(pdf_path)
        documents = loader.load()
    except Exception as e:
        print(f"Standard PDF loading failed: {e}")
        documents = []

    # Check if pages have meaningful content, if not use OCR
    has_content = any(len(doc.page_content.strip()) > 100 for doc in documents)

    if not has_content:
        print("No text found in standard extraction, trying OCR...")
        try:
            # Convert PDF pages to images and extract text using OCR
            images = convert_from_path(pdf_path)
            ocr_documents = []

            for page_num, image in enumerate(images):
                # Extract text using Tesseract OCR
                text = pytesseract.image_to_string(image)

                # Create a document for this page
                doc = Document(
                    page_content=text,
                    metadata={
                        "source": pdf_path,
                        "page": page_num,
                        "extraction_method": "OCR"
                    }
                )
                ocr_documents.append(doc)

            documents = ocr_documents
            print(f"Successfully extracted {len(documents)} pages using OCR")
        except Exception as e:
            error_msg = str(e)
            print(f"OCR extraction failed: {e}")
            if "poppler" in error_msg.lower() or "page count" in error_msg.lower():
                print("\n❌ Poppler is not installed or not in PATH!")
                print("Fix this by running: brew install poppler")
                print("\nIf Tesseract is also missing, run: brew install tesseract")
            elif "tesseract" in error_msg.lower():
                print("\n❌ Tesseract is not installed!")
                print("Fix this by running: brew install tesseract")
            else:
                print("Please ensure both Tesseract and Poppler are installed on your system.")

    return documents


/var/folders/gb/6cys8vkd0_1dzs4_k2h5pwp00000gn/T/ipykernel_36904/191032532.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, YoutubeLoader


In [3]:
# Usage
pdf_path = "/Users/00156257shakeermohammedaudhil/Documents/ZRAG/data/embedded-systems.pdf"
# pdf_path = "/Users/00156257shakeermohammedaudhil/Documents/ZRAG/data/audhil-report.pdf"
pdf_pages = load_pdf_with_ocr(pdf_path)
if pdf_pages:
    print(f"Loaded {len(pdf_pages)} pages")
    print("First page content preview:")
    print(pdf_pages[0].page_content[:1500])
    #print(pdf_pages[0])

Loaded 209 pages
First page content preview:
Embedded Systems Design: An Introduction to Processes, Tools, and 
Techniques 
by Arnold S. Berger ISBN: 1578200733 
CMP Books © 2002 (237 pages) 
An easy-to-understand guidebook for those embarking upon an embedded 
processor development project.  
 
 
Table of Contents  
 
 
Embedded Systems Design—An Introduc tion to Processes, Tools, and 
Techniques  
 Preface  
 Introduction  
 Chapter 1 - The Embedded Design Life Cycle 
 Chapter 2 - The Selection Process 
 Chapter 3 - The Partitioning Decision 
 Chapter 4 - The Development Environment 
 Chapter 5 - Special Software Techniques 
 Chapter 6 - A Basic Toolset 
 Chapter 7 - BDM, JTAG, and Nexus 
 Chapter 8 - The ICE — An Integrated Solution 
 Chapter 9 - Testing 
 Chapter 10 - The Future 
 Index  
 List of Figures  
 List of Tables  
 List of Listings  
 List of Sidebars  
 
TEAMFLY
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Tea

In [4]:
YoutubeLoader

langchain_community.document_loaders.youtube.YoutubeLoader

In [5]:
# Fix PATH for Homebrew tools (especially important for notebook environments)
import os
import subprocess

# Ensure homebrew bin is in PATH
homebrew_paths = ["/opt/homebrew/bin", "/usr/local/bin"]
current_path = os.environ.get("PATH", "").split(":")
for path in homebrew_paths:
    if path not in current_path:
        current_path.insert(0, path)
os.environ["PATH"] = ":".join(current_path)

# Install system dependencies for YouTube audio processing
!brew install ffmpeg --quiet 2>/dev/null || echo "FFmpeg may need manual install"
# Install Python dependencies
!pip install yt_dlp --quiet
!pip install --upgrade pip
!pip install pydub --quiet
!pip install faster-whisper --quiet
!pip install torch --quiet
!pip install ffmpeg-python --quiet

In [6]:
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers.audio import FasterWhisperParser  # to generate transcript from audio
from langchain_community.document_loaders.blob_loaders.youtube_audio import YoutubeAudioLoader

# Set FFmpeg path explicitly for Homebrew installation
os.environ["PATH"] = "/opt/homebrew/bin:" + os.environ.get("PATH", "")

# Also set explicit ffmpeg location for yt-dlp
import subprocess
ffmpeg_path = None
ffprobe_path = None

try:
    ffmpeg_path = subprocess.check_output(["which", "ffmpeg"], text=True).strip()
    ffprobe_path = subprocess.check_output(["which", "ffprobe"], text=True).strip()

    # Set environment variables for the entire process
    os.environ["FFMPEG_LOCATION"] = ffmpeg_path
    os.environ["FFPROBE_LOCATION"] = ffprobe_path
    os.environ["PATH"] = f"{os.path.dirname(ffmpeg_path)}:" + os.environ.get("PATH", "")

    print(f"✓ FFmpeg found at: {ffmpeg_path}")
    print(f"✓ FFprobe found at: {ffprobe_path}")

    # Create yt-dlp config to locate ffmpeg
    yt_dlp_config_dir = os.path.expanduser("~/.config/yt-dlp")
    os.makedirs(yt_dlp_config_dir, exist_ok=True)

    yt_dlp_config = os.path.join(yt_dlp_config_dir, "config.txt")
    with open(yt_dlp_config, "w") as f:
        f.write(f"# Auto-generated config\nffmpeg-location {os.path.dirname(ffmpeg_path)}\n")
    print(f"✓ yt-dlp config created at {yt_dlp_config}")

except Exception as e:
    print(f"Error: Could not locate ffmpeg/ffprobe: {e}")
    print("Please install: brew install ffmpeg")


✓ FFmpeg found at: /opt/homebrew/bin/ffmpeg
✓ FFprobe found at: /opt/homebrew/bin/ffprobe
✓ yt-dlp config created at /Users/00156257shakeermohammedaudhil/.config/yt-dlp/config.txt


In [7]:
url = "https://www.youtube.com/watch?v=uFhDGagZzjs"
save_dir = "/Users/00156257shakeermohammedaudhil/Documents/ZRAG/data/youtube"

# Create save directory if it doesn't exist
os.makedirs(save_dir, exist_ok=True)

# Load YouTube audio and transcribe
try:
    print(f"Downloading and transcribing: {url}")
    loader = GenericLoader(
        YoutubeAudioLoader([url], save_dir),
        FasterWhisperParser()
    )
    docs = loader.load()
    print(f"\n✓ Successfully loaded {len(docs)} documents from YouTube")
    if docs:
        print("\nTranscription preview:")
        print(docs[0].page_content[:500])
except Exception as e:
    error_str = str(e)
    print(f"\n❌ Error loading YouTube audio: {e}")
    print(f"FFmpeg location: {ffmpeg_path}")
    print("\nTroubleshooting:")
    print("1. Ensure FFmpeg is installed: brew install ffmpeg")
    print("2. Verify PATH: which ffmpeg, which ffprobe")
    print("3. Test FFmpeg: ffmpeg -version")
    if "ffmpeg" in error_str.lower() or "ffprobe" in error_str.lower():
        print("\nThe issue is FFmpeg-related. Try restarting the notebook.")


[youtube] Extracting URL: https://www.youtube.com/watch?v=uFhDGagZzjs
[youtube] uFhDGagZzjs: Downloading webpage


[youtube] uFhDGagZzjs: Downloading android vr player API JSON
[info] uFhDGagZzjs: Downloading 1 format(s): 140
[download] /Users/00156257shakeermohammedaudhil/Documents/ZRAG/data/youtube/Lecture 01： Introduction to Embedded Systems.m4a has already been downloaded
[download] 100% of   27.27MiB
[ExtractAudio] Not converting audio /Users/00156257shakeermohammedaudhil/Documents/ZRAG/data/youtube/Lecture 01： Introduction to Embedded Systems.m4a; file is already in target format m4a


[2026-06-07 13:37:02.980] [ctranslate2] [thread 2510691] [warning] The compute type inferred from the saved model is float16, but the target device or backend do not support efficient float16 computation. The model weights have been automatically converted to use the float32 compute type instead.



✓ Successfully loaded 219 documents from YouTube

Transcription preview:
 Let me welcome you to this course which will be conducted jointly by myself and Doctor


In [10]:
len(docs)

219

In [11]:
len(pdf_pages)

209

In [12]:
combined_docs = pdf_pages + docs
print(f"Total combined documents: {len(combined_docs)}")

Total combined documents: 428
